In [0]:
bronze = spark.read.format("delta").load("/Volumes/workspace/ecommerce/delta/bronze/events")

### Create Binary Purchase Label

In [0]:
from pyspark.sql import functions as F

label_df = bronze.groupBy("user_id") \
    .agg(F.max(
        F.when(F.col("event_type")=="purchase",1).otherwise(0)
    ).alias("purchased"))

In [0]:
features_df = spark.read.format("delta").load("/Volumes/workspace/ecommerce/delta/silver/user_features_1")

### Join with Feature Table

In [0]:
training_data = features_df.join(label_df,"user_id")

In [0]:
training_data.display()

### Split Train/Test

In [0]:
train_df, test_df = training_data.randomSplit([0.8, 0.2], seed=42)

### Validate Distribution

In [0]:
training_data.groupBy('purchased').count().show()

In [0]:
from pyspark.sql.functions import col

total = training_data.count()
positive = training_data.filter(col("purchased") == 1).count()

print("Positive rate:", round((positive / total)*100,2), "%")
